In [1]:
# ------------------------------------------------------------
# VWAP + RSI Based Intraday Option Scalper Backtester (Groww-style)
# ------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from growwapi import GrowwAPI

# -------------------- CONFIG --------------------
API_KEY = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NDcxMzEyNjAsImlhdCI6MTc1ODczMTI2MCwibmJmIjoxNzU4NzMxMjYwLCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCJkYWMzYzY1ZS0zNzYyLTQ2M2MtYTdhNS0wOTEzOTFmZWFlZDRcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiYTM3YTY5OWQtNDliNC00ODVhLWFhYjQtNjQyMjY0ZTU1ODk1XCIsXCJkZXZpY2VJZFwiOlwiNjMwZDE0OGQtMzgwOS01OTI5LTk0MmYtOTdiN2U2ODFlZmNiXCIsXCJzZXNzaW9uSWRcIjpcIjMxZGQyN2MwLTkxZWItNDQ2MS1hOGUxLTFhNjM0YmYzMzJiNVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkJtSEhnYnA4UHhkZmhvZUlwZ1JaY2hSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCIyMDIuNzEuMy4yMSwxNzIuNjkuODYuMjUwLDM1LjI0MS4yMy4xMjNcIixcInR3b0ZhRXhwaXJ5VHNcIjoyNTQ3MTMxMjYwODE4fSIsImlzcyI6ImFwZXgtYXV0aC1wcm9kLWFwcCJ9.tX5bpR3IsOdUlTB0pTazt1UrwEOjwIpi-uE7E2MgkIRlMOfq_5EshYnklO8txDKe94cN17-7NdeVk2RckgX_KA"
API_SECRET = "UNYYIEBAT2LB7XGVHRRP2QOIYHWEOZCP"

SYMBOL = "NIFTY24OCT24500CE"      # example ATM option
EXCHANGE = GrowwAPI.EXCHANGE_NSE
SEGMENT = GrowwAPI.SEGMENT_CASH
PRODUCT = GrowwAPI.PRODUCT_MIS
VALIDITY = GrowwAPI.VALIDITY_DAY
ORDER_TYPE = GrowwAPI.ORDER_TYPE_MARKET

TRADE_COST = 83        # ₹ per round trip
TARGET_POINTS = 3.0    # ≈ ₹225 profit/lot
STOPLOSS_POINTS = 2.0  # ≈ ₹150 loss/lot
QTY = 50               # lot size
COOLDOWN_MIN = 10      # min gap between trades
RSI_PERIOD = 7

# -------------------- DATA --------------------
# Load 1-min OHLC data (csv must have: Datetime, Open, High, Low, Close, Volume)
df = pd.read_csv("minute_data/NSEI_2025-10-24.csv", parse_dates=["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)

# Compute VWAP
df["CumVol"] = df["Volume"].cumsum()
df["CumVolPrice"] = (df["Close"] * df["Volume"]).cumsum()
df["VWAP"] = df["CumVolPrice"] / df["CumVol"]

# RSI Function
def compute_rsi(series, period=7):
    delta = series.diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(gain).rolling(period).mean()
    avg_loss = pd.Series(loss).rolling(period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI"] = compute_rsi(df["Close"], RSI_PERIOD)

# -------------------- BACKTEST LOOP --------------------
trades = []
position = None
entry_price = 0
last_exit_time = df["Datetime"].iloc[0] - pd.Timedelta(minutes=COOLDOWN_MIN)

for i in range(1, len(df)):
    row = df.iloc[i]
    prev = df.iloc[i - 1]

    # Skip until indicators exist
    if np.isnan(row["RSI"]) or np.isnan(row["VWAP"]):
        continue

    # Cooldown logic
    if (row["Datetime"] - last_exit_time).seconds / 60 < COOLDOWN_MIN:
        continue

    # Entry logic
    if position is None:
        # BUY setup
        if (row["Close"] > row["VWAP"]) and (row["RSI"] > 55) and (row["Close"] > prev["Close"]):
            position = "LONG"
            entry_price = row["Close"]
            entry_time = row["Datetime"]
        # SELL setup
        elif (row["Close"] < row["VWAP"]) and (row["RSI"] < 45) and (row["Close"] < prev["Close"]):
            position = "SHORT"
            entry_price = row["Close"]
            entry_time = row["Datetime"]

    # Exit logic
    elif position == "LONG":
        if row["Close"] >= entry_price + TARGET_POINTS or row["Close"] <= entry_price - STOPLOSS_POINTS:
            pnl = (row["Close"] - entry_price) * QTY - TRADE_COST
            trades.append([entry_time, row["Datetime"], "LONG", entry_price, row["Close"], pnl])
            position = None
            last_exit_time = row["Datetime"]

    elif position == "SHORT":
        if row["Close"] <= entry_price - TARGET_POINTS or row["Close"] >= entry_price + STOPLOSS_POINTS:
            pnl = (entry_price - row["Close"]) * QTY - TRADE_COST
            trades.append([entry_time, row["Datetime"], "SHORT", entry_price, row["Close"], pnl])
            position = None
            last_exit_time = row["Datetime"]

# -------------------- RESULTS --------------------
trades_df = pd.DataFrame(trades, columns=["EntryTime", "ExitTime", "Side", "EntryPrice", "ExitPrice", "PnL"])
total_pnl = trades_df["PnL"].sum()
win_rate = (trades_df["PnL"] > 0).mean() * 100
avg_pnl = trades_df["PnL"].mean()
num_trades = len(trades_df)

print("Total Trades:", num_trades)
print("Total P&L: ₹", round(total_pnl, 2))
print("Win Rate:", round(win_rate, 2), "%")
print("Average P&L per Trade: ₹", round(avg_pnl, 2))

# -------------------- PLOT --------------------
if num_trades > 0:
    trades_df["CumPnL"] = trades_df["PnL"].cumsum()
    plt.figure(figsize=(10, 5))
    plt.plot(trades_df["ExitTime"], trades_df["CumPnL"])
    plt.title("Cumulative P&L")
    plt.xlabel("Time")
    plt.ylabel("PnL (₹)")
    plt.grid(True)
    plt.show()

Total Trades: 0
Total P&L: ₹ 0
Win Rate: nan %
Average P&L per Trade: ₹ nan


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from growwapi import GrowwAPI

# -------------------- CONFIG --------------------
# -------------------- CONFIG --------------------
API_KEY = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NDcxMzEyNjAsImlhdCI6MTc1ODczMTI2MCwibmJmIjoxNzU4NzMxMjYwLCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCJkYWMzYzY1ZS0zNzYyLTQ2M2MtYTdhNS0wOTEzOTFmZWFlZDRcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiYTM3YTY5OWQtNDliNC00ODVhLWFhYjQtNjQyMjY0ZTU1ODk1XCIsXCJkZXZpY2VJZFwiOlwiNjMwZDE0OGQtMzgwOS01OTI5LTk0MmYtOTdiN2U2ODFlZmNiXCIsXCJzZXNzaW9uSWRcIjpcIjMxZGQyN2MwLTkxZWItNDQ2MS1hOGUxLTFhNjM0YmYzMzJiNVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkJtSEhnYnA4UHhkZmhvZUlwZ1JaY2hSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCIyMDIuNzEuMy4yMSwxNzIuNjkuODYuMjUwLDM1LjI0MS4yMy4xMjNcIixcInR3b0ZhRXhwaXJ5VHNcIjoyNTQ3MTMxMjYwODE4fSIsImlzcyI6ImFwZXgtYXV0aC1wcm9kLWFwcCJ9.tX5bpR3IsOdUlTB0pTazt1UrwEOjwIpi-uE7E2MgkIRlMOfq_5EshYnklO8txDKe94cN17-7NdeVk2RckgX_KA"
API_SECRET = "UNYYIEBAT2LB7XGVHRRP2QOIYHWEOZCP"
SYMBOL = "NIFTY24OCT24500CE"
TRADE_COST = 83
TARGET_POINTS = 3.0
STOPLOSS_POINTS = 2.0
QTY = 50
COOLDOWN_MIN = 3

# -------------------- DATA --------------------
df = pd.read_csv("minute_data/NSEI_2025-10-24.csv", parse_dates=["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)

# VWAP
df["CumVol"] = df["Volume"].cumsum()
df["CumVolPrice"] = (df["Close"] * df["Volume"]).cumsum()
df["VWAP"] = df["CumVolPrice"] / df["CumVol"]

# Short-term momentum
df["Mom2"] = df["Close"] - df["Close"].shift(2)

# -------------------- BACKTEST --------------------
trades = []
position = None
entry_price = 0
last_exit_time = df["Datetime"].iloc[0] - pd.Timedelta(minutes=COOLDOWN_MIN)

for i in range(3, len(df)):
    row = df.iloc[i]
    prev = df.iloc[i - 1]

    if np.isnan(row["VWAP"]):
        continue

    # Cooldown logic
    if (row["Datetime"] - last_exit_time).seconds / 60 < COOLDOWN_MIN:
        continue

    # ENTRY
    if position is None:
        # Long bias
        if (row["Close"] > row["VWAP"] * 1.001) and (row["Mom2"] > 0):
            position = "LONG"
            entry_price = row["Close"]
            entry_time = row["Datetime"]

        # Short bias
        elif (row["Close"] < row["VWAP"] * 0.999) and (row["Mom2"] < 0):
            position = "SHORT"
            entry_price = row["Close"]
            entry_time = row["Datetime"]

    # EXIT
    elif position == "LONG":
        if row["Close"] >= entry_price + TARGET_POINTS or row["Close"] <= entry_price - STOPLOSS_POINTS:
            pnl = (row["Close"] - entry_price) * QTY - TRADE_COST
            trades.append([entry_time, row["Datetime"], "LONG", entry_price, row["Close"], pnl])
            position = None
            last_exit_time = row["Datetime"]

    elif position == "SHORT":
        if row["Close"] <= entry_price - TARGET_POINTS or row["Close"] >= entry_price + STOPLOSS_POINTS:
            pnl = (entry_price - row["Close"]) * QTY - TRADE_COST
            trades.append([entry_time, row["Datetime"], "SHORT", entry_price, row["Close"], pnl])
            position = None
            last_exit_time = row["Datetime"]

# -------------------- RESULTS --------------------
trades_df = pd.DataFrame(trades, columns=["EntryTime", "ExitTime", "Side", "EntryPrice", "ExitPrice", "PnL"])
if len(trades_df) == 0:
    print("⚠️ No trades found. Try loosening thresholds or check data quality.")
else:
    total_pnl = trades_df["PnL"].sum()
    win_rate = (trades_df["PnL"] > 0).mean() * 100
    avg_pnl = trades_df["PnL"].mean()

    print(f"Total Trades: {len(trades_df)}")
    print(f"Total P&L: ₹{round(total_pnl,2)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg P&L/trade: ₹{avg_pnl:.2f}")

    trades_df["CumPnL"] = trades_df["PnL"].cumsum()
    plt.figure(figsize=(10,5))
    plt.plot(trades_df["ExitTime"], trades_df["CumPnL"])
    plt.title("Cumulative Intraday P&L")
    plt.xlabel("Time")
    plt.ylabel("PnL (₹)")
    plt.grid(True)
    plt.show()

⚠️ No trades found. Try loosening thresholds or check data quality.


In [4]:
print(df.head(5))
print(df[['Datetime', 'Close', 'Volume']].tail(5))
print("Data points:", len(df))
print("Unique minutes:", df['Datetime'].nunique())
print("Avg Volume:", df['Volume'].mean())
print("Price Range:", df['Close'].min(), "→", df['Close'].max())


                   Datetime         Close          High           Low  \
0 2025-10-24 09:15:00+05:30  25882.199219  25939.300781  25877.500000   
1 2025-10-24 09:16:00+05:30  25868.199219  25886.099609  25859.599609   
2 2025-10-24 09:17:00+05:30  25862.150391  25881.750000  25856.250000   
3 2025-10-24 09:18:00+05:30  25850.550781  25869.449219  25849.650391   
4 2025-10-24 09:19:00+05:30  25850.800781  25853.349609  25846.050781   

           Open  Volume  CumVol  CumVolPrice  VWAP       Mom2  
0  25939.300781       0       0          0.0   NaN        NaN  
1  25885.699219       0       0          0.0   NaN        NaN  
2  25867.650391       0       0          0.0   NaN -20.048828  
3  25862.050781       0       0          0.0   NaN -17.648438  
4  25849.150391       0       0          0.0   NaN -11.349609  
                     Datetime         Close  Volume
370 2025-10-24 15:25:00+05:30  25799.199219       0
371 2025-10-24 15:26:00+05:30  25803.449219       0
372 2025-10-24 15:27:

In [9]:
from growwapi import GrowwAPI
import pyotp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# -------------------- CONFIG --------------------
API_KEY = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NDcxMzEyNjAsImlhdCI6MTc1ODczMTI2MCwibmJmIjoxNzU4NzMxMjYwLCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCJkYWMzYzY1ZS0zNzYyLTQ2M2MtYTdhNS0wOTEzOTFmZWFlZDRcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiYTM3YTY5OWQtNDliNC00ODVhLWFhYjQtNjQyMjY0ZTU1ODk1XCIsXCJkZXZpY2VJZFwiOlwiNjMwZDE0OGQtMzgwOS01OTI5LTk0MmYtOTdiN2U2ODFlZmNiXCIsXCJzZXNzaW9uSWRcIjpcIjMxZGQyN2MwLTkxZWItNDQ2MS1hOGUxLTFhNjM0YmYzMzJiNVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkJtSEhnYnA4UHhkZmhvZUlwZ1JaY2hSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCIyMDIuNzEuMy4yMSwxNzIuNjkuODYuMjUwLDM1LjI0MS4yMy4xMjNcIixcInR3b0ZhRXhwaXJ5VHNcIjoyNTQ3MTMxMjYwODE4fSIsImlzcyI6ImFwZXgtYXV0aC1wcm9kLWFwcCJ9.tX5bpR3IsOdUlTB0pTazt1UrwEOjwIpi-uE7E2MgkIRlMOfq_5EshYnklO8txDKe94cN17-7NdeVk2RckgX_KA"
API_SECRET = "UNYYIEBAT2LB7XGVHRRP2QOIYHWEOZCP"
SYMBOL = "NIFTY25OCT24500CE"
TRADE_DATE = "2025-10-24"

TRADE_COST = 83
TARGET_POINTS = 3.0
STOPLOSS_POINTS = 2.0
QTY = 50
COOLDOWN_MIN = 3

# -------------------- AUTHENTICATION --------------------
def init_groww():
    totp_gen = pyotp.TOTP(API_SECRET)
    totp = totp_gen.now()
    access_token = GrowwAPI.get_access_token(API_KEY, totp)
    return GrowwAPI(access_token)

api = init_groww()
print("✅ Groww API connected!")

# -------------------- FETCH HISTORICAL DATA --------------------
try:
    end_time = datetime.strptime(TRADE_DATE, "%Y-%m-%d") + timedelta(hours=15, minutes=30)
    start_time = end_time.replace(hour=9, minute=15)

    df = api.get_historical_data(
        symbol=SYMBOL,
        interval="1m",
        from_date=start_time.strftime("%Y-%m-%d %H:%M:%S"),
        to_date=end_time.strftime("%Y-%m-%d %H:%M:%S")
    )

    df = pd.DataFrame(df)
    df["Datetime"] = pd.to_datetime(df["time"])
    df = df[["Datetime", "close", "high", "low", "open", "volume"]]
    df.columns = ["Datetime", "Close", "High", "Low", "Open", "Volume"]

    print(f"✅ Loaded {len(df)} rows of 1-min data for {SYMBOL} ({TRADE_DATE})")

except Exception as e:
    print("❌ Error fetching data:", e)

Ready to Groww!
✅ Groww API connected!
❌ Error fetching data: 'GrowwAPI' object has no attribute 'get_historical_data'


In [18]:
# ======================================================
# Groww API: Fetch Historical Candle Data + VWAP Backtest
# ======================================================

from growwapi import GrowwAPI
import pyotp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# -------------------- CONFIG --------------------
API_KEY = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NDcxMzEyNjAsImlhdCI6MTc1ODczMTI2MCwibmJmIjoxNzU4NzMxMjYwLCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCJkYWMzYzY1ZS0zNzYyLTQ2M2MtYTdhNS0wOTEzOTFmZWFlZDRcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiYTM3YTY5OWQtNDliNC00ODVhLWFhYjQtNjQyMjY0ZTU1ODk1XCIsXCJkZXZpY2VJZFwiOlwiNjMwZDE0OGQtMzgwOS01OTI5LTk0MmYtOTdiN2U2ODFlZmNiXCIsXCJzZXNzaW9uSWRcIjpcIjMxZGQyN2MwLTkxZWItNDQ2MS1hOGUxLTFhNjM0YmYzMzJiNVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkJtSEhnYnA4UHhkZmhvZUlwZ1JaY2hSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCIyMDIuNzEuMy4yMSwxNzIuNjkuODYuMjUwLDM1LjI0MS4yMy4xMjNcIixcInR3b0ZhRXhwaXJ5VHNcIjoyNTQ3MTMxMjYwODE4fSIsImlzcyI6ImFwZXgtYXV0aC1wcm9kLWFwcCJ9.tX5bpR3IsOdUlTB0pTazt1UrwEOjwIpi-uE7E2MgkIRlMOfq_5EshYnklO8txDKe94cN17-7NdeVk2RckgX_KA"
API_SECRET = "UNYYIEBAT2LB7XGVHRRP2QOIYHWEOZCP"

SYMBOL = "NIFTY25N0426500CE"
EXCHANGE = "NSE"
SEGMENT = "FNO"

START_TIME = "2025-10-27 10:00:00"
END_TIME = "2025-10-27 14:00:00"
INTERVAL = 1  # in minutes (1 or 5 etc.)

# -------------------- AUTHENTICATION --------------------
def init_groww():
    """Authenticate with Groww using TOTP (Time-based OTP)."""
    try:
        totp = pyotp.TOTP(API_SECRET).now()
        access_token = GrowwAPI.get_access_token(API_KEY, totp)
        groww = GrowwAPI(access_token)
        print("✅ Groww API connected successfully!")
        return groww
    except Exception as e:
        print("❌ Groww authentication failed:", e)
        return None

groww = init_groww()

# -------------------- FETCH HISTORICAL DATA --------------------
def fetch_candles(symbol, exchange, segment, start_time, end_time, interval):
    try:
        response = groww.get_historical_candle_data(
            trading_symbol=symbol,
            exchange=getattr(groww, f"EXCHANGE_{exchange.upper()}"),
            segment=getattr(groww, f"SEGMENT_{segment.upper()}"),
            start_time=start_time,
            end_time=end_time,
            interval_in_minutes=interval
        )
        candles = response.get("data", [])
        if not candles:
            print("⚠️ No data returned from Groww API.")
            return pd.DataFrame()

        df = pd.DataFrame(candles)
        df.columns = ["Datetime", "Open", "High", "Low", "Close", "Volume"]
        df["Datetime"] = pd.to_datetime(df["Datetime"])
        df = df.sort_values("Datetime").reset_index(drop=True)
        print(f"✅ Loaded {len(df)} candles ({interval}-min)")
        return df

    except Exception as e:
        print("❌ Error fetching historical data:", e)
        return pd.DataFrame()

df = fetch_candles(SYMBOL, EXCHANGE, SEGMENT, START_TIME, END_TIME, INTERVAL)

# -------------------- VISUALIZE --------------------
if not df.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(df["Datetime"], df["Close"], label=f"{SYMBOL} Close")
    plt.title(f"{SYMBOL} ({INTERVAL}-min candles)")
    plt.xlabel("Time")
    plt.ylabel("Price (₹)")
    plt.legend()
    plt.grid(True)
    plt.show()

# -------------------- (OPTIONAL) SIMPLE VWAP LOGIC --------------------
if not df.empty:
    df["CumVol"] = df["Volume"].cumsum()
    df["CumVolPrice"] = (df["Close"] * df["Volume"]).cumsum()
    df["VWAP"] = df["CumVolPrice"] / df["CumVol"]
    df["Mom2"] = df["Close"] - df["Close"].shift(2)

    plt.figure(figsize=(10, 5))
    plt.plot(df["Datetime"], df["Close"], label="Close")
    plt.plot(df["Datetime"], df["VWAP"], label="VWAP", linestyle="--")
    plt.title("VWAP Indicator")
    plt.legend()
    plt.show()

Ready to Groww!
✅ Groww API connected successfully!
⚠️ No data returned from Groww API.


In [15]:
# Optional: timeout parameter (in seconds) for the API call; default is typically set by the SDK.
holdings_response = groww.get_holdings_for_user(timeout=5)
print(holdings_response)

{'holdings': [{'isin': 'INE155A01022', 'trading_symbol': 'TMPV', 'quantity': 9.0, 'average_price': 412.35, 'pledge_quantity': 0.0, 'demat_locked_quantity': 0.0, 'groww_locked_quantity': 0.0, 'repledge_quantity': 0.0, 't1_quantity': 9.0, 'demat_free_quantity': 0.0, 'corporate_action_additional_quantity': 0, 'active_demat_transfer_quantity': 0, 'tradable_exchanges': ['NSE', 'BSE']}, {'isin': 'INF204KB17I5', 'trading_symbol': 'GOLDBEES', 'quantity': 200.0, 'average_price': 91.86, 'pledge_quantity': 0.0, 'demat_locked_quantity': 0.0, 'groww_locked_quantity': 0.0, 'repledge_quantity': 0.0, 't1_quantity': 0.0, 'demat_free_quantity': 200.0, 'corporate_action_additional_quantity': 0, 'active_demat_transfer_quantity': 0, 'tradable_exchanges': ['NSE', 'BSE']}]}


In [21]:
user_positions_response = groww.get_positions_for_user() # returns positions of both CASH and FNO segment.

 
print(user_positions_response)

{'positions': [{'trading_symbol': 'NIFTY25N0426500CE', 'segment': 'FNO', 'credit_quantity': 75, 'credit_price': 30.8, 'debit_quantity': 75, 'debit_price': 33.45, 'carry_forward_credit_quantity': 0, 'carry_forward_credit_price': 0.0, 'carry_forward_debit_quantity': 0, 'carry_forward_debit_price': 0.0, 'exchange': 'NSE', 'symbol_isin': 'NIFTY25N0426500CE', 'quantity': 0, 'product': 'NRML', 'net_carry_forward_quantity': 0, 'net_price': 33.45, 'net_carry_forward_price': 0.0, 'realised_pnl': 198.75}]}
